In [1]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    for path in [Path.cwd(), *Path.cwd().parents]:
        if (path / "pyproject.toml").exists() and (path / "outputs").exists():
            return path
    raise RuntimeError("Could not find beyond-click-sim repo root")


REPO_ROOT = find_repo_root()

sys.path.insert(0, str(REPO_ROOT))

In [2]:
REPO_ROOT

PosixPath('/Users/a.agafonov/Research/agent_simulator/repos/beyond-click-sim-worktrees/listwise-ranking')

In [3]:
from __future__ import annotations

from pathlib import Path

from beyond_click_sim.llm_clients import make_llm_client
from beyond_click_sim.scorers import LLMInteractionListwiseRankingScorer
from beyond_click_sim.tasks import Task
from runners.in_distribution.llm_item_stats import (
    item_rating_column_labels,
    maybe_add_item_rating_prompt_columns,
)
from runners.in_distribution.interaction_prediction.methods.common import (
    candidate_group_summary,
    current_git_commit,
    limit_candidate_groups,
    ranking_metrics_for_split,
    ranking_metrics_with_failed_groups_as_zero,
    score_coverage_summary,
    score_frame,
    task_xy,
    write_json,
)
from runners.in_distribution.interaction_prediction.metrics import (
    RANKING_KS,
    RANKING_MAIN_METRIC,
    RANKING_METRICS_FILENAME,
    RANKING_TIE_POLICY,
)
from runners.in_distribution.interaction_prediction.methods.llm_yes_no import (
    DATASET_JSON_LIST_COLUMNS,
    DATASET_PROMPT_COLUMNS,
    QWEN_EXTRA_BODY,
    _score_groups,
    _write_errors,
)
from runners.in_distribution.interaction_prediction.task_builders import repo_root

In [4]:
dataset_name = "ml-1m"

In [11]:
prompt_columns = maybe_add_item_rating_prompt_columns(
    dataset_name,
    DATASET_PROMPT_COLUMNS[dataset_name],
    use_item_stats=True,
)

prompt_columns

{'history_description_columns': ('item_title',
  'item_genres',
  'rating',
  'item_rating_mean',
  'item_rating_count'),
 'candidate_description_columns': ('item_title',
  'item_genres',
  'item_rating_mean',
  'item_rating_count')}

In [10]:
column_labels = item_rating_column_labels(
    dataset_name,
    use_item_stats=True,
)

column_labels

{'rating': 'user rating',
 'item_rating_mean': 'average rating',
 'item_rating_count': 'number of prior reviews'}